In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Find the dataset in the project folder
project_folder = Path.cwd()

csv_files = [
    file for file in project_folder.rglob("*.csv")
    if "turkish" in file.name.lower()
]

if not csv_files:
    raise FileNotFoundError(
        "Dataset not found. Make sure the CSV is inside the project folder."
    )

file_path = csv_files[0]
print("Dataset file:", file_path)

# Load the semicolon-separated dataset
df = pd.read_csv(file_path, sep=";", encoding="utf-8-sig")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset file: C:\Users\kavya\Turkish Crowdfunding Startups\dataset\turkish+crowdfunding+startups\turkishCF.csv
Dataset loaded successfully!
Dataset shape: (1628, 38)


,id,platform_adi,kitle_fonlamasi_turu,kategori,fon_sekli,proje_adi,proje_sahibi,proje_sahibi_cinsiyet,kac_proje_destekledi,kac_projeye_abone,...,sosyal_medya,sm_sayisi,sm_takipci,etiket_sayisi,icerik_kelime_sayisi,proje_aciklamasi,hedef_miktari,toplanan_tutar,destek_orani,basari_durumu
0,1,fongogo,ödül,diğer,ya hep ya hiç,Gerçek Gizlidir Filmleri,Lob Ekibi,belirsiz,1,0,...,var,3,274,0,301,Türkiye’de ilk kez her aşaması ‘online’ olarak...,40000,54410,136%,başarılı
1,2,fongogo,ödül,diğer,ya hep ya hiç,Fongogo - Hayat Bulsun!,Fongogo Team,belirsiz,3,0,...,var,3,5634,0,167,"Fongogo ekibi olarak büyümek, daha geniş kitle...",50000,50110,100%,başarılı
2,3,fongogo,ödül,diğer,ya hep ya hiç,"PAYLAŞIMIN, SEVGİNİN VE ÇOCUKLARIN İYİLİK EVİ ...",SİNEM CAN,kadın,0,0,...,var,2,510,4,413,Çocuklara her şeyin ücretsiz olduğu iyilik evi...,47000,47850,101%,başarılı
3,4,fongogo,ödül,diğer,ya hep ya hiç,Türk Kadının 100.yıl Seyri,Deniz Tutkusu Seyirde,kadın,0,0,...,var,2,3338,5,287,‘Sözde değil Özde İzindeyiz ‘ sloganıyla Atatü...,35000,35450,101%,başarılı
4,5,fongogo,ödül,diğer,ya hep ya hiç,Doğadaki Çocuk,Seren Kutadgu,kadın,2,0,...,var,1,144,3,571,İstanbul'daki çocuk yuvalarında yaşayan dezava...,30000,31250,104%,başarılı


In [3]:
# Select relevant features for clustering
selected_features = [
    "platform_adi",
    "kitle_fonlamasi_turu",
    "kategori",
    "fon_sekli",
    "proje_sahibi_cinsiyet",
    "kac_proje_destekledi",
    "kac_projeye_abone",
    "kac_projenin_sahibi",
    "kac_proje_takiminda",
    "konum",
    "bolge",
    "yil",
    "gun_sayisi",
    "tanitim_videosu",
    "video_uzunlugu",
    "gorsel_sayisi",
    "sss",
    "guncellemeler",
    "yorumlar",
    "odul_sayisi",
    "ekip_kisi_sayisi",
    "web_sitesi",
    "sosyal_medya",
    "sm_sayisi",
    "sm_takipci",
    "etiket_sayisi",
    "icerik_kelime_sayisi",
    "hedef_miktari"
]

# Keep only columns that exist in the dataset
selected_features = [
    column for column in selected_features
    if column in df.columns
]

cluster_df = df[selected_features].copy()

print("Selected features:", len(selected_features))
print("Selected columns:")
print(selected_features)

print("\nClustering dataset shape:", cluster_df.shape)
display(cluster_df.head())

Selected features: 28
Selected columns:
['platform_adi', 'kitle_fonlamasi_turu', 'kategori', 'fon_sekli', 'proje_sahibi_cinsiyet', 'kac_proje_destekledi', 'kac_projeye_abone', 'kac_projenin_sahibi', 'kac_proje_takiminda', 'konum', 'bolge', 'yil', 'gun_sayisi', 'tanitim_videosu', 'video_uzunlugu', 'gorsel_sayisi', 'sss', 'guncellemeler', 'yorumlar', 'odul_sayisi', 'ekip_kisi_sayisi', 'web_sitesi', 'sosyal_medya', 'sm_sayisi', 'sm_takipci', 'etiket_sayisi', 'icerik_kelime_sayisi', 'hedef_miktari']

Clustering dataset shape: (1628, 28)


,platform_adi,kitle_fonlamasi_turu,kategori,fon_sekli,proje_sahibi_cinsiyet,kac_proje_destekledi,kac_projeye_abone,kac_projenin_sahibi,kac_proje_takiminda,konum,...,yorumlar,odul_sayisi,ekip_kisi_sayisi,web_sitesi,sosyal_medya,sm_sayisi,sm_takipci,etiket_sayisi,icerik_kelime_sayisi,hedef_miktari
0,fongogo,ödül,diğer,ya hep ya hiç,belirsiz,1,0,1,0,istanbul,...,0,11,4,yok,var,3,274,0,301,40000
1,fongogo,ödül,diğer,ya hep ya hiç,belirsiz,3,0,2,0,istanbul,...,0,11,7,yok,var,3,5634,0,167,50000
2,fongogo,ödül,diğer,ya hep ya hiç,kadın,0,0,2,1,belirsiz,...,1,6,2,var,var,2,510,4,413,47000
3,fongogo,ödül,diğer,ya hep ya hiç,kadın,0,0,1,0,muğla,...,0,7,1,var,var,2,3338,5,287,35000
4,fongogo,ödül,diğer,ya hep ya hiç,kadın,2,0,1,0,istanbul,...,0,8,6,yok,var,1,144,3,571,30000


In [4]:
# Identify numerical and categorical columns in the selected features
numerical_columns = cluster_df.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = cluster_df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:", len(numerical_columns))
print(numerical_columns)

print("\nCategorical features:", len(categorical_columns))
print(categorical_columns)

Numerical features: 18
['kac_proje_destekledi', 'kac_projeye_abone', 'kac_projenin_sahibi', 'kac_proje_takiminda', 'yil', 'gun_sayisi', 'video_uzunlugu', 'gorsel_sayisi', 'sss', 'guncellemeler', 'yorumlar', 'odul_sayisi', 'ekip_kisi_sayisi', 'sm_sayisi', 'sm_takipci', 'etiket_sayisi', 'icerik_kelime_sayisi', 'hedef_miktari']

Categorical features: 10
['platform_adi', 'kitle_fonlamasi_turu', 'kategori', 'fon_sekli', 'proje_sahibi_cinsiyet', 'konum', 'bolge', 'tanitim_videosu', 'web_sitesi', 'sosyal_medya']


In [5]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Preprocessing pipeline:
# - One-hot encode categorical features
# - Standardize numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
        ("numerical", StandardScaler(), numerical_columns)
    ],
    remainder="drop"
)

X_processed = preprocessor.fit_transform(cluster_df)

# Convert sparse output to a dense array if needed
if hasattr(X_processed, "toarray"):
    X_processed = X_processed.toarray()

print("Preprocessing completed!")
print("Processed data shape:", X_processed.shape)

Preprocessing completed!
Processed data shape: (1628, 130)


In [6]:
from pathlib import Path

output_folder = Path("processed_data")
output_folder.mkdir(exist_ok=True)

# Save the processed feature matrix
processed_df = pd.DataFrame(X_processed)
output_path = output_folder / "turkish_crowdfunding_preprocessed.csv"

processed_df.to_csv(output_path, index=False)

print("Preprocessed dataset saved successfully!")
print("Saved file:", output_path)
print("Final shape:", processed_df.shape)

Preprocessed dataset saved successfully!
Saved file: processed_data\turkish_crowdfunding_preprocessed.csv
Final shape: (1628, 130)


## Result

The Turkish Crowdfunding Startups dataset was preprocessed for clustering. Relevant project features were selected, categorical variables were encoded using One-Hot Encoding, and numerical features were standardized using StandardScaler. The processed feature matrix was saved as `turkish_crowdfunding_preprocessed.csv`.

## Conclusion

The dataset was successfully transformed into a numerical format suitable for unsupervised machine learning. The encoded and scaled data is ready for feature preparation and clustering algorithms such as K-Means.